# Submission Builder v2

**Key changes from v1:**
- **75 trees** instead of 150 — reduces judge latency ~3x, giving much better latency penalty
- **No threshold tuning** — v1 thresholds overfitted on OOF folds; raw argmax is more robust
- Encoding maps consistent with inference (full-train encoding)

| | v1 | v2 |
|---|---|---|
| Trees | 150 | **75** |
| Threshold tuning | Yes (overfitted) | **No** |
| Judge latency (est.) | 2.8s | **~1.0s** |
| Latency penalty | 0.717 | **~0.90** |
| Estimated score | 0.333 | **~0.52** |

## 1. Imports & Paths

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib, os, zipfile, shutil, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
import warnings; warnings.filterwarnings('ignore')

SEED, SMOOTHING = 42, 300
DATA_DIR = '../Data'
SUB_DIR  = '../submission_v2'
os.makedirs(SUB_DIR, exist_ok=True)

TARGET = 'Purchased_Coverage_Bundle'
N_TREES = 75           # Optimal from benchmark
JUDGE_FACTOR = 6.9     # Judge is ~6.9x slower than local

print('LightGBM:', lgb.__version__)
print('N_TREES:', N_TREES)

LightGBM: 4.6.0
N_TREES: 75


## 2. Load Data

In [2]:
train     = pd.read_csv(os.path.join(DATA_DIR, 'train_clean.csv'))
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

X = train.drop(columns=[TARGET])
y = train[TARGET]
FEATURE_COLS = X.columns.tolist()

print('Train clean:', train.shape)
print('Test  raw:  ', test_raw.shape)
print('Features:   ', len(FEATURE_COLS))
print()
print('Class distribution:')
print(y.value_counts().sort_index().to_string())

Train clean: (60868, 45)
Test  raw:   (15218, 28)
Features:    44

Class distribution:
Purchased_Coverage_Bundle
0      823
1     1625
2    36136
3     4831
4    13958
5      479
6      719
7     2286
8        6
9        5


## 3. Full-Train Target Encoding Maps

Used by both training (replacing OOF-encoded columns) and inference. Consistent encoding.

In [3]:
GLOBAL_MEAN = float(train_raw[TARGET].mean())

def build_enc_map(series, target_series, smoothing=SMOOTHING):
    df_tmp = pd.DataFrame({'cat': series.fillna(-1), 'tgt': target_series})
    agg = df_tmp.groupby('cat')['tgt'].agg(['mean', 'count'])
    agg['enc'] = (agg['count']*agg['mean'] + smoothing*GLOBAL_MEAN) / (agg['count']+smoothing)
    return agg['enc'].to_dict()

BROKER_ENC_MAP   = build_enc_map(train_raw['Broker_ID'],   train_raw[TARGET])
REGION_ENC_MAP   = build_enc_map(train_raw['Region_Code'], train_raw[TARGET])
EMPLOYER_ENC_MAP = build_enc_map(train_raw['Employer_ID'], train_raw[TARGET])

print(f'Global mean: {GLOBAL_MEAN:.4f}')
print(f'Broker  map: {len(BROKER_ENC_MAP)} entries')
print(f'Region  map: {len(REGION_ENC_MAP)} entries')
print(f'Employer map: {len(EMPLOYER_ENC_MAP)} entries')

Global mean: 2.7441
Broker  map: 316 entries
Region  map: 167 entries
Employer map: 310 entries


## 4. Quick OOF Validation (3-fold)

Fast sanity check before full training.

In [4]:
def resample(X_tr, y_tr, seed=SEED):
    vc = pd.Series(y_tr).value_counts()
    ros_s = {c: 50 for c in [8, 9] if c in vc.index and vc[c] < 50}
    if ros_s:
        X_tr, y_tr = RandomOverSampler(sampling_strategy=ros_s, random_state=seed).fit_resample(X_tr, y_tr)
    vc = pd.Series(y_tr).value_counts()
    sm_s = {c: 2000 for c in [0, 5, 6] if c in vc.index and vc[c] < 2000}
    if sm_s:
        X_tr, y_tr = SMOTE(sampling_strategy=sm_s, k_neighbors=5, random_state=seed).fit_resample(X_tr, y_tr)
    return X_tr, y_tr

lgbm_params = dict(
    objective='multiclass', num_class=10, metric='multi_logloss',
    class_weight='balanced', n_estimators=N_TREES, learning_rate=0.1,
    max_depth=7, num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=1, verbose=-1,
)

Xv = X.values
yv = y.values
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(yv), dtype=int)

for fold, (tr_idx, val_idx) in enumerate(kf.split(Xv, yv)):
    X_tr, y_tr = resample(Xv[tr_idx], yv[tr_idx], seed=SEED+fold)
    clf = lgb.LGBMClassifier(**lgbm_params)
    clf.fit(X_tr, y_tr)
    oof_preds[val_idx] = clf.predict(Xv[val_idx])
    f = f1_score(yv[val_idx], oof_preds[val_idx], average='macro', zero_division=0)
    print(f'  Fold {fold}: F1={f:.4f}')

oof_f1 = f1_score(yv, oof_preds, average='macro', zero_division=0)
print(f'\nOOF Macro F1 ({N_TREES} trees, raw argmax): {oof_f1:.4f}')

  Fold 0: F1=0.6013


  Fold 1: F1=0.5625


  Fold 2: F1=0.5992

OOF Macro F1 (75 trees, raw argmax): 0.5827


## 5. Train Final Model on Full Data

In [5]:
X_bal, y_bal = resample(Xv, yv)
print('After resampling:', pd.Series(y_bal).value_counts().sort_index().to_string())

final_model = lgb.LGBMClassifier(**lgbm_params)
final_model.fit(X_bal, y_bal)
booster = final_model.booster_

print(f'\nTrained: {N_TREES} trees × 10 classes = {booster.num_trees()} total trees')

After resampling: 0     2000
1     1625
2    36136
3     4831
4    13958
5     2000
6     2000
7     2286
8       50
9       50



Trained: 75 trees × 10 classes = 750 total trees


## 6. Save `model.pkl`

In [6]:
MODEL_PATH = os.path.join(SUB_DIR, 'model.pkl')

payload = {
    'model_str':        booster.model_to_string(),
    'feature_cols':     FEATURE_COLS,       # NO thresholds — raw argmax
    'broker_enc_map':   BROKER_ENC_MAP,
    'region_enc_map':   REGION_ENC_MAP,
    'employer_enc_map': EMPLOYER_ENC_MAP,
    'global_mean':      GLOBAL_MEAN,
}
joblib.dump(payload, MODEL_PATH, compress=3)

size_mb = os.path.getsize(MODEL_PATH) / (1024*1024)
print(f'Saved model.pkl -> {size_mb:.2f} MB')
print(f'Size penalty: max(0.5, 1-{size_mb:.2f}/200) = {max(0.5,1-size_mb/200):.4f}')

Saved model.pkl -> 1.60 MB
Size penalty: max(0.5, 1-1.60/200) = 0.9920


## 7. Write `solution.py`

In [7]:
SOL_PATH = os.path.join(SUB_DIR, 'solution.py')
shutil.copy('/tmp/solution_v2.py', SOL_PATH)
print('Copied solution_v2.py ->', SOL_PATH)
print(f'Lines: {len(open(SOL_PATH).readlines())}')

Copied solution_v2.py -> ../submission_v2/solution.py
Lines: 159


## 8. Write `requirements.txt`

In [8]:
REQ_PATH = os.path.join(SUB_DIR, 'requirements.txt')
with open(REQ_PATH, 'w') as f:
    f.write('# All required packages are pre-installed in the judge environment.\n')
    f.write('# lightgbm==4.6.0\n')
    f.write('# numpy==1.26.4\n')
    f.write('# pandas==2.1.4\n')
    f.write('# scikit-learn==1.3.2\n')
    f.write('# joblib==1.3.2\n')
print('Written requirements.txt')

Written requirements.txt


## 9. Build `submission_v2.zip`

In [9]:
ZIP_PATH = '../submission_v2.zip'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.write(SOL_PATH,   'solution.py')
    zf.write(MODEL_PATH, 'model.pkl')
    zf.write(REQ_PATH,   'requirements.txt')

zip_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
print(f'submission_v2.zip -> {zip_mb:.2f} MB')
print('Contents:')
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        print(f'  {info.filename:<22s}  {info.file_size/1024:.1f} KB')
print(f'\nSize < 50 MB: {zip_mb < 50}')

submission_v2.zip -> 1.60 MB
Contents:
  solution.py             6.5 KB
  model.pkl               1639.2 KB
  requirements.txt        0.2 KB

Size < 50 MB: True


## 10. Validate — Simulate Judge Pipeline

In [10]:
import importlib.util

spec = importlib.util.spec_from_file_location('solution_v2', SOL_PATH)
sol  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)

# 1. Preprocess
df_proc = sol.preprocess(test_raw.copy())
print(f'Preprocessed shape: {df_proc.shape}  |  User_ID present: {"User_ID" in df_proc.columns}')

# 2. Load model (not timed)
t0 = time.perf_counter()
loaded = sol.load_model()
print(f'Load time: {time.perf_counter()-t0:.3f} s (not scored)')

# 3. Predict (timed — 5 runs)
runs = []
for _ in range(5):
    t0 = time.perf_counter()
    out = sol.predict(df_proc.copy(), loaded)
    runs.append(time.perf_counter()-t0)
lat = np.median(runs)

print(f'Predict latency: {lat:.3f} s  |  runs: {[f"{r:.3f}" for r in runs]}')
print(f'Output shape: {out.shape}')
print(f'Dtype: {out["Purchased_Coverage_Bundle"].dtype}')
print(f'Unique predictions: {sorted(out["Purchased_Coverage_Bundle"].unique())}')
print()
print(out.head(8).to_string())

Preprocessed shape: (15218, 51)  |  User_ID present: True
Load time: 0.046 s (not scored)


Predict latency: 0.191 s  |  runs: ['0.188', '0.205', '0.195', '0.191', '0.180']
Output shape: (15218, 2)
Dtype: int64
Unique predictions: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

      User_ID  Purchased_Coverage_Bundle
0  USR_060868                          0
1  USR_060869                          2
2  USR_060870                          2
3  USR_060871                          4
4  USR_060872                          2
5  USR_060873                          4
6  USR_060874                          3
7  USR_060875                          7


## 11. Final Score Estimate

In [11]:
judge_lat = lat * JUDGE_FACTOR
s_pen     = max(0.5, 1 - zip_mb / 200)
l_pen     = max(0.5, 1 - judge_lat / 10)
est       = oof_f1 * s_pen * l_pen

print('=' * 60)
print('  SUBMISSION v2 — ESTIMATED COMPETITION SCORE')
print('=' * 60)
print(f'  OOF Macro F1:      {oof_f1:.5f} (3-fold, {N_TREES} trees, no thresholds)')
print(f'  ZIP size:          {zip_mb:.2f} MB   (pen: {s_pen:.4f})')
print(f'  Local latency:     {lat:.3f} s')
print(f'  Judge lat (×{JUDGE_FACTOR}):  {judge_lat:.2f} s   (pen: {l_pen:.4f})')
print(f'  EST. FINAL SCORE:  {est:.5f}')
print('=' * 60)
print()
print('  v1 actual:  score=0.3331  F1=0.4719  lat=2.83s  pen=0.717')
print(f'  v2 est:     score={est:.4f}  F1={oof_f1:.4f}  lat={judge_lat:.2f}s  pen={l_pen:.3f}')
print()
print(f'Submission ZIP: {os.path.abspath(ZIP_PATH)}')

  SUBMISSION v2 — ESTIMATED COMPETITION SCORE
  OOF Macro F1:      0.58271 (3-fold, 75 trees, no thresholds)
  ZIP size:          1.60 MB   (pen: 0.9920)
  Local latency:     0.191 s
  Judge lat (×6.9):  1.32 s   (pen: 0.8680)
  EST. FINAL SCORE:  0.50176

  v1 actual:  score=0.3331  F1=0.4719  lat=2.83s  pen=0.717
  v2 est:     score=0.5018  F1=0.5827  lat=1.32s  pen=0.868

Submission ZIP: /home/tesla/Desktop/DataQuest/DataQuest/submission_v2.zip
